# Nordstar Customer 360 — Bronze → Silver → Gold Pipeline
**Kata series 7.W** · DuckDB + Python · local Jupyter

This notebook implements all seven katas.  
Each section is self-contained: run cells top-to-bottom within a section.

In [1]:
# ── Cell 1: Environment setup (kata 7.W.0) ────────────────────────────────────
# Compatible with local Jupyter and Google Colab.
# If ModuleNotFoundError after install: restart kernel, run this cell first.

import subprocess, sys

# Install / upgrade dependencies in the active Python environment
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "duckdb>=1.4", "pandas", "--quiet"]
)

import duckdb          # columnar in-process OLAP engine
import pandas as pd    # DataFrame for display and light manipulation
import os              # file path helpers
import random          # synthetic data generation
import numpy as np     # numerical helpers
from datetime import datetime  # timestamp construction

# Create an in-memory DuckDB connection.
# All tables live in RAM; nothing is written to disk until we COPY or EXPORT.
con = duckdb.connect()

# ── Hello-world smoke test ─────────────────────────────────────────────────────
con.execute("""
    CREATE TABLE hello_world (
        id          INTEGER,
        message     VARCHAR,
        created_at  TIMESTAMP
    )
""")

con.execute("""
    INSERT INTO hello_world VALUES
        (1, 'Bronze layer ready',  '2026-07-28 09:00:00'),
        (2, 'Silver layer ready',  '2026-07-28 09:01:00'),
        (3, 'Gold layer ready',    '2026-07-28 09:02:00')
""")

result = con.execute("SELECT * FROM hello_world ORDER BY id").df()
display(result)

print("Environment ready ✓")
print(f"  python  : {sys.version.split()[0]}")
print(f"  duckdb  : {duckdb.__version__}")
print(f"  pandas  : {pd.__version__}")

,id,message,created_at
0,1,Bronze layer ready,2026-07-28 09:00:00
1,2,Silver layer ready,2026-07-28 09:01:00
2,3,Gold layer ready,2026-07-28 09:02:00


Environment ready ✓
  python  : 3.14.3
  duckdb  : 1.5.5
  pandas  : 3.0.5


In [2]:

# ── Cell 2: Bronze layer — synthetic raw data (kata 7.W.1) ────────────────────
# Generates bronze/transactions_raw.csv with deliberate quality issues.

import os, random
import numpy as np
import pandas as pd
from datetime import date, timedelta

random.seed(42)
np.random.seed(42)

N = 500
BRONZE_DIR = "bronze"
os.makedirs(BRONZE_DIR, exist_ok=True)

# ── base columns ──────────────────────────────────────────────────────────────
order_ids = [f"ORD-{random.randint(10000,99999):05d}" for _ in range(N)]

# 3% duplicates: replace ~15 rows with an already-used order_id
dup_count = int(N * 0.03)
dup_sources = [order_ids[i] for i in random.sample(range(N - dup_count), dup_count)]
for i, src in zip(random.sample(range(N), dup_count), dup_sources):
    order_ids[i] = src

customer_ids = [random.randint(1000, 9999) for _ in range(N)]

regions = random.choices(["North", "South", "East", "West"], k=N)

categories = random.choices(
    ["Electronics", "Clothing", "Food", "Home", "Sports"], k=N
)

# ── mixed date formats ────────────────────────────────────────────────────────
start = date(2024, 1, 1)
raw_dates = [start + timedelta(days=random.randint(0, 365)) for _ in range(N)]

def fmt_date(d, style):
    if style == 0:
        return d.strftime("%Y-%m-%d")       # ISO:  2024-01-15
    elif style == 1:
        return d.strftime("%d/%m/%Y")       # EU:   15/01/2024
    else:
        return d.strftime("%b %d %Y")       # Text: Jan 15 2024

date_styles = random.choices([0, 1, 2], weights=[0.5, 0.3, 0.2], k=N)
order_dates = [fmt_date(d, s) for d, s in zip(raw_dates, date_styles)]

# ── amount: 5% null, 2% negative ─────────────────────────────────────────────
amounts = np.round(np.random.uniform(5.0, 500.0, N), 2)
null_idx = random.sample(range(N), int(N * 0.05))
for i in null_idx:
    amounts[i] = np.nan
neg_idx = random.sample([i for i in range(N) if i not in null_idx], int(N * 0.02))
for i in neg_idx:
    amounts[i] = -abs(amounts[i])

quantities = [random.randint(1, 10) for _ in range(N)]

statuses = random.choices(
    ["completed", "returned", "pending"], weights=[0.80, 0.15, 0.05], k=N
)

# ── assemble DataFrame and save ───────────────────────────────────────────────
df = pd.DataFrame({
    "order_id":         order_ids,
    "customer_id":      customer_ids,
    "region":           regions,
    "order_date":       order_dates,
    "product_category": categories,
    "amount":           amounts,
    "quantity":         quantities,
    "status":           statuses,
})

out_path = os.path.join(BRONZE_DIR, "transactions_raw.csv")
df.to_csv(out_path, index=False)

# ── quick stats ───────────────────────────────────────────────────────────────
null_amount  = df["amount"].isna().sum()
dup_order_ids = df["order_id"].duplicated(keep=False).sum()
date_formats = set()
for d in df["order_date"]:
    if "/" in d:
        date_formats.add("DD/MM/YYYY")
    elif d[0:3].isalpha():
        date_formats.add("Mon DD YYYY")
    else:
        date_formats.add("YYYY-MM-DD")

print(f"Saved  : {out_path}")
print(f"Rows   : {len(df)}")
print(f"Nulls in amount     : {null_amount}  (target ≈ 25)")
print(f"Duplicate order_ids : {dup_order_ids}  (target ≈ 15)")
print(f"Date formats found  : {sorted(date_formats)}")


Saved  : bronze/transactions_raw.csv
Rows   : 500
Nulls in amount     : 25  (target ≈ 25)
Duplicate order_ids : 35  (target ≈ 15)
Date formats found  : ['DD/MM/YYYY', 'Mon DD YYYY', 'YYYY-MM-DD']


In [ ]:

# ── Cell 3: Silver layer — cleaning (kata 7.W.2) ──────────────────────────────
# Bronze profile baseline: 500 rows, 25 null amounts, 18 duplicate order_ids.
# Rules:
#   1. Drop rows where amount IS NULL.
#   2. Standardise order_date → DATE (3 input formats).
#   3. Deduplicate by order_id, keep highest customer_id.
#   4. Retain negative amounts (valid returns).

import os, duckdb

os.makedirs("silver", exist_ok=True)
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE TABLE silver AS
WITH bronze_raw AS (
    SELECT *
    FROM read_csv_auto('bronze/transactions_raw.csv', ignore_errors=true)
),

-- Step 1: drop null amounts (25 rows)
not_null AS (
    SELECT * FROM bronze_raw
    WHERE amount IS NOT NULL
),

-- Step 2: standardise order_date across all three input formats
date_parsed AS (
    SELECT
        order_id,
        customer_id,
        region,
        COALESCE(
            TRY_STRPTIME(order_date, '%Y-%m-%d'),   -- 2024-01-15
            TRY_STRPTIME(order_date, '%d/%m/%Y'),   -- 15/01/2024
            TRY_STRPTIME(order_date, '%b %d %Y')    -- Jan 15 2024
        )::DATE AS order_date,
        product_category,
        CAST(amount AS DOUBLE) AS amount,
        quantity,
        status
    FROM not_null
),

-- Step 3: deduplicate — keep highest customer_id per order_id
deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY customer_id DESC
        ) AS rn
    FROM date_parsed
)
SELECT order_id, customer_id, region, order_date,
       product_category, amount, quantity, status
FROM deduped
WHERE rn = 1
""")

# Write to Parquet
con.execute("""
    COPY silver TO 'silver/transactions_clean.parquet' (FORMAT PARQUET)
""")

# Verification query
ver = con.execute("""
    SELECT
        COUNT(*)                             AS silver_rows,
        COUNT(*) - COUNT(amount)             AS null_amount,
        COUNT(*) - COUNT(DISTINCT order_id)  AS duplicate_order_ids,
        COUNT(CASE WHEN amount < 0 THEN 1 END) AS negative_amounts
    FROM 'silver/transactions_clean.parquet'
""").df()

print("=== Silver verification ===")
print(ver.T.to_string())
print()

# Manual math check
bronze_rows   = 500
null_removed  = 25
silver_actual = ver["silver_rows"].iloc[0]
dup_removed   = bronze_rows - null_removed - silver_actual
print(f"Bronze rows      : {bronze_rows}")
print(f"Null rows removed: {null_removed}")
print(f"Dup rows removed : {dup_removed}  (from {18} extra copies in bronze)")
print(f"Expected silver  : {bronze_rows - null_removed} - {dup_removed} = {bronze_rows - null_removed - dup_removed}")
print(f"Actual silver    : {silver_actual}")
delta_pct = abs(silver_actual - (bronze_rows - null_removed)) / (bronze_rows - null_removed) * 100
print(f"Delta from null-only estimate: {delta_pct:.1f}%  (threshold 10%)")


=== Silver verification ===
                       0
silver_rows          459
null_amount            0
duplicate_order_ids    0
negative_amounts       9

Bronze rows      : 500
Null rows removed: 25
Dup rows removed : 16  (from 18 extra copies in bronze)
Expected silver  : 475 - 16 = 459
Actual silver    : 459
Delta from null-only estimate: 3.4%  (threshold 10%)


In [ ]:
# ── Cell 4: Gold layer — business metrics (kata 7.W.3) ───────────────────────
import duckdb, os

os.makedirs("gold", exist_ok=True)
con = duckdb.connect()

# ── Table 1: daily_sales_by_category ─────────────────────────────────────────
# Grain: one row per (order_date, region, product_category)
# total_revenue: completed orders only (negatives are returns, not revenue)
# order_count:  distinct completed order_ids
con.execute("""
CREATE OR REPLACE TABLE daily_sales_by_category AS
SELECT
    order_date,
    region,
    product_category,
    SUM(CASE WHEN status = 'completed' THEN amount ELSE 0 END)        AS total_revenue,
    COUNT(DISTINCT CASE WHEN status = 'completed' THEN order_id END)  AS order_count
FROM 'silver/transactions_clean.parquet'
GROUP BY order_date, region, product_category
ORDER BY order_date, region, product_category
""")

con.execute("COPY daily_sales_by_category TO 'gold/daily_sales_by_category.parquet' (FORMAT PARQUET)")

# ── Table 2: returns_rate ─────────────────────────────────────────────────────
# Denominator: completed + returned (exclude pending — not yet finalised)
# COALESCE: all-pending dates → 0.0 instead of NULL
con.execute("""
CREATE OR REPLACE TABLE returns_rate AS
SELECT
    order_date,
    COUNT(CASE WHEN status IN ('completed', 'returned') THEN 1 END)  AS total_orders,
    COUNT(CASE WHEN status = 'returned' THEN 1 END)                    AS returned_orders,
    COALESCE(ROUND(
        COUNT(CASE WHEN status = 'returned' THEN 1 END)
        / NULLIF(COUNT(CASE WHEN status IN ('completed', 'returned') THEN 1 END), 0)
        * 100,
    2), 0.0)                                                             AS returns_rate_pct
FROM 'silver/transactions_clean.parquet'
GROUP BY order_date
ORDER BY order_date
""")

con.execute("COPY returns_rate TO 'gold/returns_rate.parquet' (FORMAT PARQUET)")

# ── Row counts ────────────────────────────────────────────────────────────────
r1 = con.execute("SELECT COUNT(*) FROM 'gold/daily_sales_by_category.parquet'").fetchone()[0]
r2 = con.execute("SELECT COUNT(*) FROM 'gold/returns_rate.parquet'").fetchone()[0]
print(f"daily_sales_by_category rows : {r1}")
print(f"returns_rate rows            : {r2}")

# ── Grain check ───────────────────────────────────────────────────────────────
grain = con.execute("""
    SELECT COUNT(*) AS total,
           COUNT(DISTINCT order_date || '|' || region || '|' || product_category) AS unique_combos
    FROM 'gold/daily_sales_by_category.parquet'
""").df()
print("\n=== Grain check ===")
total, unique = int(grain["total"].iloc[0]), int(grain["unique_combos"].iloc[0])
print(f"  total={total}  unique_combos={unique}  {'PASS' if total==unique else 'FAIL'}")

# ── Range check ───────────────────────────────────────────────────────────────
rng = con.execute("""
    SELECT MIN(returns_rate_pct), MAX(returns_rate_pct),
           COUNT(*) FILTER (WHERE returns_rate_pct IS NULL) AS null_count
    FROM 'gold/returns_rate.parquet'
""").fetchone()
print(f"\n=== returns_rate_pct range ===")
print(f"  min={rng[0]}  max={rng[1]}  nulls={rng[2]}  {'PASS' if rng[0]>=0 and rng[1]<=100 and rng[2]==0 else 'FAIL'}")


daily_sales_by_category rows : 443
returns_rate rows            : 260

=== Grain check ===
  total=443  unique_combos=443  PASS

=== returns_rate_pct range ===
  min=0.0  max=100.0  nulls=0  PASS


In [ ]:
# ── Cell 5: DQ checks — break-and-verify (kata 7.W.4) ────────────────────────
import duckdb

con = duckdb.connect()
con.execute("CREATE OR REPLACE TABLE daily_sales  AS SELECT * FROM 'gold/daily_sales_by_category.parquet'")
con.execute("CREATE OR REPLACE TABLE returns_rate AS SELECT * FROM 'gold/returns_rate.parquet'")

def check(name, sql, con):
    n = con.execute(sql).fetchone()[0]
    print(f"  {'✓' if n==0 else '✗'}  {name}: {'PASS' if n==0 else f'FAIL  [{n} violating row(s)]'}")
    return n == 0

def run_all_checks(con, label=""):
    if label:
        print(f"\n{'='*58}\n  DQ run: {label}\n{'='*58}")
    passed  = 0
    passed += check("1  No null key cols (daily_sales)",
        "SELECT COUNT(*) FROM daily_sales WHERE order_date IS NULL OR region IS NULL OR product_category IS NULL", con)
    passed += check("2  total_revenue > 0",
        "SELECT COUNT(*) FROM daily_sales WHERE total_revenue <= 0", con)
    passed += check("3  order_count > 0",
        "SELECT COUNT(*) FROM daily_sales WHERE order_count <= 0", con)
    passed += check("4  No duplicate grain (date·region·category)",
        "SELECT COUNT(*)-COUNT(DISTINCT order_date||'|'||region||'|'||product_category) FROM daily_sales", con)
    passed += check("5  No null order_date (returns_rate)",
        "SELECT COUNT(*) FROM returns_rate WHERE order_date IS NULL", con)
    passed += check("6  returns_rate_pct in [0, 100]",
        "SELECT COUNT(*) FROM returns_rate WHERE returns_rate_pct < 0 OR returns_rate_pct > 100 OR returns_rate_pct IS NULL", con)
    passed += check("7  returned_orders <= total_orders",
        "SELECT COUNT(*) FROM returns_rate WHERE returned_orders > total_orders", con)
    passed += check("8  Date range spans >= 30 days",
        "SELECT CASE WHEN MAX(order_date)-MIN(order_date)>=30 THEN 0 ELSE 1 END FROM returns_rate", con)
    print(f"\n  {passed}/8 checks passed.")
    return passed

run_all_checks(con, "clean gold data (baseline)")

# Inject 3 bad rows to prove checks fire
con.execute("""INSERT INTO daily_sales VALUES
    ('2024-06-15', 'North', 'Electronics', -999.99, 3),
    ('2024-06-15', 'North', 'Electronics',  150.00, 2),
    (NULL,          'South', 'Clothing',       75.00, 1)""")
run_all_checks(con, "after bad-row injection (expect 3 FAIL)")

# Cleanup
con.execute("DELETE FROM daily_sales WHERE total_revenue < 0 OR order_date IS NULL")
con.execute("DELETE FROM daily_sales WHERE order_date='2024-06-15' AND region='North' AND product_category='Electronics' AND total_revenue=150.00")
run_all_checks(con, "after cleanup (expect 8/8)")



  DQ run: clean gold data (baseline)
  ✓  1  No null key cols (daily_sales): PASS
  ✓  2  total_revenue > 0: PASS
  ✓  3  order_count > 0: PASS
  ✓  4  No duplicate grain (date·region·category): PASS
  ✓  5  No null order_date (returns_rate): PASS
  ✓  6  returns_rate_pct in [0, 100]: PASS
  ✓  7  returned_orders <= total_orders: PASS
  ✓  8  Date range spans >= 30 days: PASS

  8/8 checks passed.

  DQ run: after bad-row injection (expect 3 FAIL)
  ✗  1  No null key cols (daily_sales): FAIL  [1 violating row(s)]
  ✗  2  total_revenue > 0: FAIL  [1 violating row(s)]
  ✓  3  order_count > 0: PASS
  ✗  4  No duplicate grain (date·region·category): FAIL  [2 violating row(s)]
  ✓  5  No null order_date (returns_rate): PASS
  ✓  6  returns_rate_pct in [0, 100]: PASS
  ✓  7  returned_orders <= total_orders: PASS
  ✓  8  Date range spans >= 30 days: PASS

  5/8 checks passed.

  DQ run: after cleanup (expect 8/8)
  ✓  1  No null key cols (daily_sales): PASS
  ✓  2  total_revenue > 0: PASS
  

In [ ]:
# ── Cell 6: Dashboard — inline charts + Streamlit (kata 7.W.5) ───────────────
# Inline matplotlib equivalents verify real data loads.
# Streamlit version: kata-workspace/app.py  (run: python3 -m streamlit run app.py)
#
# One thing I would change: add a 7-day rolling average to the returns-rate line
# chart — the daily values are too spiky (0↔100%) to show the trend to a manager.

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

sales   = pd.read_parquet('gold/daily_sales_by_category.parquet')
returns = pd.read_parquet('gold/returns_rate.parquet')
sales['order_date']   = pd.to_datetime(sales['order_date'])
returns['order_date'] = pd.to_datetime(returns['order_date'])

print(f"sales rows: {len(sales)}  |  regions: {sorted(sales['region'].unique())}")
print(f"returns rows: {len(returns)}  |  pct range: [{returns['returns_rate_pct'].min():.1f}, {returns['returns_rate_pct'].max():.1f}]")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: revenue by region, grouped by category
rev = sales.groupby(['region', 'product_category'])['total_revenue'].sum().unstack(fill_value=0)
rev.plot(kind='bar', ax=axes[0], width=0.7)
axes[0].set_title('Revenue by Region & Category')
axes[0].set_xlabel('Region'); axes[0].set_ylabel('Revenue ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Category', fontsize=8)

# Chart 2: returns rate over time
ret_sorted = returns.sort_values('order_date')
axes[1].plot(ret_sorted['order_date'], ret_sorted['returns_rate_pct'], color='#ef553b', linewidth=0.8)
axes[1].set_title('Returns Rate Over Time')
axes[1].set_xlabel('Date'); axes[1].set_ylabel('Returns Rate (%)')
axes[1].set_ylim(0, 110)

plt.tight_layout()
plt.savefig('gold/inline_charts.png', dpi=120)
print('Charts saved → gold/inline_charts.png')
display(plt.gcf())


sales rows: 351  |  regions: ['East', 'North', 'South', 'West']
returns rows: 260  |  pct range: [0.0, 100.0]
Charts saved → gold/inline_charts.png
